In [ ]:
!pip install transformers
!pip install pyannote.audio

In [1]:
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from pyannote.audio import Pipeline
from whisperx_numpy2_compatibility import load_align_model, align
from whisperx_numpy2_compatibility.diarize import assign_word_speakers

import textwrap
import os
import logging

#os.environ['CURL_CA_BUNDLE'] = ''

INFO:speechbrain.utils.quirks:Applied quirks (see `speechbrain.utils.quirks`): [allow_tf32, disable_jit_profiling]
INFO:speechbrain.utils.quirks:Excluded quirks specified by the `SB_DISABLE_QUIRKS` environment (comma-separated list): []


In [2]:
def find_intersections(speakers, texts):
    intersections = []

    for text in texts:
        text_start, text_end = text['start'], text['end']-0.1
        for turn, _, speaker in speakers.itertracks(yield_label=True):
            speaker_start, speaker_end = turn.start, turn.end
            
            # Find the overlap between the speaker's interval and the text's interval
            start = max(text_start, speaker_start)
            end = min(text_end, speaker_end)
            
            if start < end:  # There is an intersection
                if intersections and intersections[-1]['speaker'] == speaker:
                    intersections[-1]['end'] = end
                    intersections[-1]['text'] += ' ' + text['text']
                else:
                    intersections.append({
                        'start': start,
                        'end': end,
                        'speaker': speaker,
                        'text': text['text']
                    })
    return intersections


In [3]:
LOCAL_MODEL = False

In [4]:
def merge_speech_segments(segments):
    merged_segments = []
    for segment in segments:
        if merged_segments and segment["speaker"] == merged_segments[-1]["speaker"]:
            # Extend the end time and append text for the same speaker
            merged_segments[-1]["end"] = segment["end"]
            merged_segments[-1]["text"] += " " + segment["text"]
        else:
            # Add a new segment if the speaker changes
            merged_segments.append(segment)
    return merged_segments


In [5]:
def save_speech_to_file_with_indent(segments, filename):
    with open(filename, "w", encoding="utf-8") as file:
        for segment in segments:
            # Format the speaker tag
            speaker_tag = f"{segment['speaker'].upper()}:\n"
            
            # Wrap the text to 128 characters and indent each line
            wrapped_text = textwrap.fill(segment["text"], width=128, subsequent_indent="    ")
            
            # Write the formatted text to the file
            file.write(speaker_tag)
            file.write(wrapped_text)
            file.write("\n\n")  # Add a blank line between speakers


In [6]:
HF_TOKEN="XXXXXX"

if LOCAL_MODEL:
    DIARIZATION_MODEL="/Projects/AI/models/speaker-diarization-3.1/config.yaml"
    align_model="/Projects/AI/models/wav2vec2-large-xlsr-53-russian/"
else:
    DIARIZATION_MODEL="pyannote/speaker-diarization-3.1"
    align_model='jonatasgrosman/wav2vec2-large-xlsr-53-russian'

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
print(device)

cuda


In [8]:
#Initializing up wisper pipeline
whisper_model_id="openai/whisper-large-v3"
whisper_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    whisper_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
whisper_model.to(device)
whisper_processor = AutoProcessor.from_pretrained(whisper_model_id)
whisper_pipe = pipeline(
    "automatic-speech-recognition",
    model=whisper_model,
    tokenizer=whisper_processor.tokenizer,
    feature_extractor=whisper_processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
)


Device set to use cuda


In [9]:
diarization_pipeline = Pipeline.from_pretrained("pyannote/speaker-diarization-3.1")
if torch.cuda.is_available():
    diarization_pipeline.to(torch.device("cuda"))
#model = whisper.load_model(WHISPER_MODEL, download_root='./models', device=DEVICE)

In [10]:
def transcript(file_name):
    logging.info('started')
    script = whisper_pipe(file_name, generate_kwargs={"language": "russian", "return_timestamps":"True"})
    with open('script_2.txt', "w", encoding="utf-8") as f:
        f.write(script["text"])    
    logging.info('loaded')
    diarized = diarization_pipeline(file_name, min_speakers=5, max_speakers=9)
    logging.info(diarized)
    model_a, metadata = load_align_model(language_code=script["language"], device=device, model_name=align_model)
    script_aligned = align(script["segments"], model_a, metadata, file_name, device)
    result_segments, word_seg = list(assign_word_speakers(
        diarized, script_aligned    
    ).values())

    transcribed = []
    for result_segment in result_segments:
        transcribed.append(
            {
                "start": result_segment["start"],
                "end": result_segment["end"],
                "text": result_segment["text"],
                "speaker": result_segment["speaker"] if 'speaker' in result_segment else "ND"
            }
        )

    merged = merge_speech_segments(transcribed)

    out_file, _ = os.path.splitext(file_name)
    out_file = f"{out_file}_transcript_2.txt"
    save_speech_to_file_with_indent(merged, out_file)

In [11]:
audios=["./audio/audio1266668284.mp3"]#, "./audio/audio1415011527.m4a", "./audio/audio1499365096.m4a"]

In [ ]:
for audio in audios:
        transcript(audio)

INFO:root:started
/opt/conda/lib/python3.12/site-packages/transformers/models/whisper/generation_whisper.py:512: FutureWarning: The input name `inputs` is deprecated. Please make sure to use `input_features` instead.
  warnings.warn(
You have passed language=russian, but also have set `forced_decoder_ids` to [[1, None], [2, 50360]] which creates a conflict. `forced_decoder_ids` will be ignored in favor of language=russian.
INFO:root:loaded
/opt/conda/lib/python3.12/site-packages/pyannote/audio/utils/reproducibility.py:74: ReproducibilityWarning: TensorFloat-32 (TF32) has been disabled as it might lead to reproducibility issues and lower accuracy.
It can be re-enabled by calling
   >>> import torch
   >>> torch.backends.cuda.matmul.allow_tf32 = True
   >>> torch.backends.cudnn.allow_tf32 = True
See https://github.com/pyannote/pyannote-audio/issues/1370 for more details.

  warnings.warn(
/opt/conda/lib/python3.12/site-packages/pyannote/audio/models/blocks/pooling.py:104: UserWarning: st